# Day 1 Sprint — SkillGraph Adaptive LLM

Run order:
1. Setup (GPU + deps)
2. Optional tiny three-model HF run (needs `HF_TOKEN`)
3. TRL GRPO training (GPU)
4. Display plots

In [ ]:
!nvidia-smi || echo "No GPU detected — switch Runtime to T4 GPU"

In [ ]:
REPO_URL = "https://github.com/Diyakalra1/skillgraph-adaptive-llm.git"
BRANCH = "main"
REPO_DIR = "/content/repo"  # Colab working copy (run setup cell before training cells)

In [ ]:
import os
import subprocess
import sys

!rm -rf {REPO_DIR}
!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "skillgraph_adaptive_env/training/requirements-trl.txt"]
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "skillgraph_adaptive_env[training]"]
)

import importlib

mod = importlib.import_module("skillgraph_adaptive_env.training.run_training_three_models")
print("import OK:", mod.__file__)

## Optional: tiny three-model HF run

1. Colab sidebar → **Secrets** → add key **`HF_TOKEN`** (your Hugging Face token).
2. Turn **Notebook access** ON for that secret.
3. Colab does **not** put secrets in `os.environ` automatically — the cell below loads them via `userdata`.
4. **Run the setup cell above first** (clone + `pip install -e`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/repo")
if not (REPO_DIR / "skillgraph_adaptive_env").is_dir():
    raise RuntimeError(
        "Repo not cloned. Run all cells from the top (setup installs the package)."
    )
os.chdir(REPO_DIR)


def _load_hf_token() -> str:
    token = os.getenv("HF_TOKEN", "").strip()
    if token:
        return token
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN").strip()
    except Exception as exc:
        raise RuntimeError(
            "HF_TOKEN missing. In Colab: Secrets → add HF_TOKEN → enable Notebook access, "
            "then re-run this cell."
        ) from exc
    os.environ["HF_TOKEN"] = token
    return token


_load_hf_token()

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "skillgraph_adaptive_env.training.run_training_three_models",
        "--episodes",
        "3",
        "--seed",
        "7",
        "--hf-token",
        os.environ["HF_TOKEN"],
        "--max-tokens",
        "64",
        "--max-api-calls",
        "40",
        "--out-dir",
        "training/runs/hf_three_models_day1",
    ],
    cwd=str(REPO_DIR),
)

## TRL GRPO (main RL proof run)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/repo")
if not (REPO_DIR / "skillgraph_adaptive_env").is_dir():
    raise RuntimeError("Run the setup cell first (clone + pip install).")
os.chdir(REPO_DIR)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "skillgraph_adaptive_env.training.run_training_trl_grpo",
        "--episodes",
        "30",
        "--seed",
        "7",
        "--max-turns",
        "12",
        "--model-id",
        "Qwen/Qwen2.5-0.5B-Instruct",
        "--max-samples",
        "90",
        "--epochs",
        "1",
        "--out-dir",
        "training/runs/trl_grpo_day1",
    ],
    cwd=str(REPO_DIR),
)

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

repo = Path("/content/repo")
if not repo.is_dir():
    repo = Path.cwd()
out = repo / "training/runs/trl_grpo_day1"
print(json.dumps(json.loads((out / "summary.json").read_text()), indent=2))
print(json.dumps(json.loads((out / "eval_summary.json").read_text()), indent=2))

for name in ["reward_vs_steps.png", "success_rate_trend.png", "reward_components.png", "training_loss.png"]:
    p = out / "plots" / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))
    else:
        print("Missing:", name)